# Phylogenetic Exploration — PEP725 Fruit Trees

Explores the phylogenetic relationships among the fruit tree and shrub species in the PEP725 dataset, and asks whether **more closely related species also share similar flowering phenology**.

The analysis uses `PhylogenyFeatures`, which:
1. Resolves scientific names via the **OpenTree TNRS** API
2. Fetches the **induced phylogenetic subtree** from OpenTree
3. Computes an **all-pairs patristic distance matrix** from the Newick tree
4. Embeds species in 2D via **classical MDS**

> **Requires internet access** for the OpenTree API calls in §4.

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
from matplotlib.gridspec import GridSpec
import seaborn as sns
from scipy.cluster import hierarchy as sch
from scipy.spatial.distance import squareform

from pysephone.dataset.dataset import Dataset
from pysephone.dataset.util.phylogeny import PhylogenyFeatures
from pysephone.constants import KEY_DATA_SOURCE, KEY_SPECIES_ID, KEY_OBSERVATIONS, KEY_OBS_TYPE

plt.rcParams.update({
    'figure.dpi': 120,
    'font.size': 10,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

## 1. Load dataset

We use `PEP725_fruit_trees_5`, a core subset covering six well-represented species: Apple, Pear, Peach, Cherry, Plum, and Blackthorn.

In [ ]:
# No calendar or feature providers needed — we work directly with observation dates
obs = Dataset.load('PEP725_fruit_trees_5').observations

print(f'Total samples (unique loc/year/species/subgroup): {len(obs)}')
print(f'Observation types: {sorted(obs.observation_types)}')
print(f'Species (src, id): {sorted(obs.species_names.keys())}')
print()
print('Species names:')
for key, name in sorted(obs.species_names.items()):
    print(f'  {key[1]:>4d}  {name}')

## 2. Flowering phenology overview

Extract first-bloom observations (`BBCH_60`) and compute day-of-year statistics per species.

In [ ]:
TARGET_OBS = 'BBCH_60'   # first bloom

# Pull observations DataFrame and filter to bloom
df_raw = obs._df_y.copy()
df_bloom = df_raw.xs(TARGET_OBS, level=KEY_OBS_TYPE).copy()
df_bloom['doy'] = df_bloom[KEY_OBSERVATIONS].dt.dayofyear
df_bloom = df_bloom.reset_index()

# Attach species names
df_bloom['species_name'] = df_bloom[KEY_SPECIES_ID].map(
    {sid: name for (src, sid), name in obs.species_names.items()}
)

summary = (
    df_bloom.groupby('species_name')['doy']
    .agg(n='count', mean='mean', std='std', q25=lambda x: x.quantile(0.25),
         median='median', q75=lambda x: x.quantile(0.75))
    .round(1)
    .sort_values('median')
)
print(summary.to_string())

In [ ]:
# Colour palette — one colour per species, ordered by median bloom DOY
species_order = summary.index.tolist()
palette = dict(zip(species_order, sns.color_palette('tab10', len(species_order))))

# DOY → approximate month label for axis
_DOY_MONTH = [(1,'Jan'),(32,'Feb'),(60,'Mar'),(91,'Apr'),(121,'May'),(152,'Jun')]

fig, ax = plt.subplots(figsize=(12, 5))

parts = ax.violinplot(
    [df_bloom.loc[df_bloom['species_name'] == sp, 'doy'].values for sp in species_order],
    positions=range(len(species_order)),
    widths=0.7,
    showmedians=True,
    showextrema=False,
)
for i, (pc, sp) in enumerate(zip(parts['bodies'], species_order)):
    pc.set_facecolor(palette[sp])
    pc.set_alpha(0.7)
parts['cmedians'].set_color('black')
parts['cmedians'].set_linewidth(1.5)

# Overlay individual points (jittered)
rng = np.random.default_rng(0)
for i, sp in enumerate(species_order):
    doys = df_bloom.loc[df_bloom['species_name'] == sp, 'doy'].values
    jitter = rng.uniform(-0.18, 0.18, len(doys))
    ax.scatter(i + jitter, doys, s=2, color=palette[sp], alpha=0.25, linewidths=0)

ax.set_xticks(range(len(species_order)))
ax.set_xticklabels([f'{sp}\n(n={summary.loc[sp,"n"]:.0f})' for sp in species_order],
                   fontsize=8.5)
ax.set_ylabel('Day of year (BBCH_60 first bloom)', fontsize=10)
ax.set_title('Flowering phenology distribution by species — PEP725 fruit trees', fontsize=12, fontweight='bold')

for doy, month in _DOY_MONTH:
    ax.axhline(doy, color='lightgrey', lw=0.6, zorder=0)
    ax.text(len(species_order) - 0.5, doy, month, fontsize=7.5, va='center', color='grey')

ax.set_xlim(-0.6, len(species_order) - 0.4)
plt.tight_layout()
plt.show()

## 3. Temporal trends per species

Mean bloom DOY per species per decade — do species track each other over time?

In [ ]:
df_bloom['year'] = df_bloom['year'].astype(int)

fig, ax = plt.subplots(figsize=(12, 5))

for sp in species_order:
    sub = df_bloom[df_bloom['species_name'] == sp].copy()
    annual = sub.groupby('year')['doy'].median()
    # Rolling 5-year median
    rolled = annual.rolling(5, center=True, min_periods=2).median()
    ax.plot(rolled.index, rolled.values, color=palette[sp], lw=1.8, label=sp)

ax.set_xlabel('Year', fontsize=10)
ax.set_ylabel('Median bloom DOY (5-yr rolling)', fontsize=10)
ax.set_title('Long-term flowering trends by species', fontsize=12, fontweight='bold')
ax.legend(fontsize=8, framealpha=0.8, ncols=2)

for doy, month in _DOY_MONTH:
    ax.axhline(doy, color='lightgrey', lw=0.6, zorder=0)

plt.tight_layout()
plt.show()

## 4. Fit phylogenetic embedding

`PhylogenyFeatures.fit()` makes two network calls:
1. **TNRS** — resolve scientific names to OpenTree taxon IDs
2. **induced_subtree** — fetch the Newick string for the matched taxa

The patristic distance matrix and MDS coordinates are then computed locally.

In [ ]:
phylo = PhylogenyFeatures(k_embed=2, output=['mds', 'distances'])

with warnings.catch_warnings(record=True) as captured:
    warnings.simplefilter('always')
    phylo.fit(obs)

if captured:
    print('Warnings during fit:')
    for w in captured:
        print(' ', str(w.message))

# Build a clean label mapping: species_key → short display name
labels = {k: obs.species_names[k] for k in phylo.species_keys}
label_list = [labels[k] for k in phylo.species_keys]   # same order as distance matrix

print('\nResolved species (in distance matrix order):')
for i, (k, name) in enumerate(zip(phylo.species_keys, label_list)):
    print(f'  [{i}]  species_id={k[1]:>3d}  {name}')

## 5. Patristic distance matrix

Heatmap of branch-length distances from the OpenTree induced subtree.
Shorter distance = more closely related.

In [ ]:
D = phylo.distance_matrix
n_sp = len(label_list)

fig, ax = plt.subplots(figsize=(8, 6.5))

# Normalise for colour mapping; zero diagonal is white
D_plot = D.copy()
mask = np.eye(n_sp, dtype=bool)

im = ax.imshow(D_plot, cmap='viridis_r', aspect='auto')
cbar = plt.colorbar(im, ax=ax, shrink=0.8)
cbar.set_label('Patristic distance (branch lengths)', fontsize=9)

ax.set_xticks(range(n_sp))
ax.set_yticks(range(n_sp))
ax.set_xticklabels(label_list, rotation=40, ha='right', fontsize=9)
ax.set_yticklabels(label_list, fontsize=9)

# Annotate cells with values
for i in range(n_sp):
    for j in range(n_sp):
        val = D[i, j]
        text_color = 'white' if val > D.max() * 0.5 else 'black'
        ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                fontsize=7.5, color=text_color)

ax.set_title('Phylogenetic distance matrix\n(patristic distances from OpenTree)', 
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Dendrogram from distance matrix

UPGMA clustering of the patristic distance matrix — this approximates the
phylogenetic tree topology from the pairwise distances.

In [ ]:
# Condensed distance vector for scipy
condensed = squareform(D, checks=False)
linkage = sch.linkage(condensed, method='average')   # UPGMA

# Colours: one per genus
def _genus(name):
    return name.split()[0]

genera = sorted(set(_genus(n) for n in label_list))
genus_colors = dict(zip(genera, sns.color_palette('Set2', len(genera))))
leaf_colors = [genus_colors[_genus(n)] for n in label_list]

fig, ax = plt.subplots(figsize=(10, 5))

dend = sch.dendrogram(
    linkage,
    labels=label_list,
    leaf_rotation=40,
    leaf_font_size=9.5,
    ax=ax,
    color_threshold=0,    # disable automatic colouring
    above_threshold_color='#555555',
)

# Colour the leaf labels by genus
xlbls = ax.get_xmajorticklabels()
for lbl in xlbls:
    genus = _genus(lbl.get_text())
    lbl.set_color(genus_colors.get(genus, 'black'))
    lbl.set_fontweight('semibold')

ax.set_ylabel('Patristic distance', fontsize=10)
ax.set_title('Phylogenetic dendrogram — PEP725 fruit tree species\n(UPGMA from OpenTree patristic distances)',
             fontsize=11, fontweight='bold')

# Genus legend
patches = [mpatches.Patch(color=c, label=g) for g, c in genus_colors.items()]
ax.legend(handles=patches, title='Genus', fontsize=8.5, title_fontsize=9,
          loc='upper right', framealpha=0.9)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['bottom'].set_visible(False)
plt.tight_layout()
plt.show()

## 7. MDS embedding

Classical MDS projects the distance matrix into 2D, preserving pairwise distances as faithfully as possible. Species close in the embedding share more evolutionary history.

In [ ]:
coords = phylo.mds_coords      # (N, 2)

# Compute variance explained by each MDS axis
D2 = D ** 2
n = D.shape[0]
H = np.eye(n) - np.ones((n, n)) / n
B = -0.5 * H @ D2 @ H
B = (B + B.T) / 2
evals = np.linalg.eigvalsh(B)
evals_pos = np.maximum(evals, 0)
evals_sorted = np.sort(evals_pos)[::-1]
var_exp = evals_sorted[:2] / evals_sorted.sum() * 100

# Mean bloom DOY per species (in distance matrix order)
mean_doy = {}
for k, name in labels.items():
    sid = k[1]
    doys = df_bloom.loc[df_bloom[KEY_SPECIES_ID] == sid, 'doy'].values
    mean_doy[name] = doys.mean() if len(doys) > 0 else np.nan

doy_vals = np.array([mean_doy.get(name, np.nan) for name in label_list])

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
fig.suptitle('MDS embedding of phylogenetic distances', fontsize=12, fontweight='bold')

# Left: coloured by genus
ax = axes[0]
for name, (x, y) in zip(label_list, coords):
    genus = _genus(name)
    color = genus_colors[genus]
    ax.scatter(x, y, s=160, color=color, edgecolors='white', linewidths=1.2, zorder=3)
    ax.annotate(
        name.split()[-1],   # epithet only for brevity
        (x, y), xytext=(6, 4), textcoords='offset points',
        fontsize=8, color=color, fontweight='semibold',
    )
ax.set_xlabel(f'MDS 1  ({var_exp[0]:.1f}% variance)', fontsize=10)
ax.set_ylabel(f'MDS 2  ({var_exp[1]:.1f}% variance)', fontsize=10)
ax.set_title('Coloured by genus', fontsize=10)
patches = [mpatches.Patch(color=c, label=g) for g, c in genus_colors.items()]
ax.legend(handles=patches, title='Genus', fontsize=8, title_fontsize=8.5, framealpha=0.9)

# Right: coloured by mean bloom DOY
ax = axes[1]
valid = ~np.isnan(doy_vals)
norm = mcolors.Normalize(vmin=doy_vals[valid].min(), vmax=doy_vals[valid].max())
cmap = plt.cm.RdYlGn_r

sc = ax.scatter(coords[:, 0], coords[:, 1],
                s=160, c=doy_vals, cmap=cmap, norm=norm,
                edgecolors='white', linewidths=1.2, zorder=3)
for name, (x, y), doy in zip(label_list, coords, doy_vals):
    ax.annotate(
        name.split()[-1],
        (x, y), xytext=(6, 4), textcoords='offset points',
        fontsize=8, color='#333333',
    )
cbar = plt.colorbar(sc, ax=ax, shrink=0.8)

# Annotate colourbar ticks with month names
tick_doys = [60, 91, 121, 152]
tick_months = ['Mar 1', 'Apr 1', 'May 1', 'Jun 1']
valid_ticks = [(d, m) for d, m in zip(tick_doys, tick_months)
               if doy_vals[valid].min() <= d <= doy_vals[valid].max()]
if valid_ticks:
    cbar.set_ticks([d for d, _ in valid_ticks])
    cbar.set_ticklabels([m for _, m in valid_ticks])
cbar.set_label('Mean bloom date', fontsize=9)

ax.set_xlabel(f'MDS 1  ({var_exp[0]:.1f}% variance)', fontsize=10)
ax.set_ylabel(f'MDS 2  ({var_exp[1]:.1f}% variance)', fontsize=10)
ax.set_title('Coloured by mean bloom date', fontsize=10)

for ax in axes:
    ax.axhline(0, color='lightgrey', lw=0.7, zorder=0)
    ax.axvline(0, color='lightgrey', lw=0.7, zorder=0)

plt.tight_layout()
plt.show()

## 8. Phylogenetic signal in phenology

Is there a **phylogenetic signal** in bloom timing?

We test this by plotting **pairwise phenological distance** (absolute difference
in mean bloom DOY) against **pairwise phylogenetic distance**.  If closely related
species also flower at similar times, we expect a positive association.

In [ ]:
# Build pairwise arrays (upper triangle, excluding diagonal)
phylo_dist_pairs = []
pheno_dist_pairs = []
pair_labels = []

for i in range(n_sp):
    for j in range(i + 1, n_sp):
        name_i = label_list[i]
        name_j = label_list[j]
        pd_val  = D[i, j]
        phen_d  = abs(mean_doy.get(name_i, np.nan) - mean_doy.get(name_j, np.nan))
        if np.isnan(pd_val) or np.isnan(phen_d):
            continue
        phylo_dist_pairs.append(pd_val)
        pheno_dist_pairs.append(phen_d)
        pair_labels.append((name_i, name_j))

phylo_dist_pairs = np.array(phylo_dist_pairs)
pheno_dist_pairs = np.array(pheno_dist_pairs)

# Pearson correlation
r = np.corrcoef(phylo_dist_pairs, pheno_dist_pairs)[0, 1]

fig, ax = plt.subplots(figsize=(7, 5.5))

# Scatter with genus-based colouring of pairs (colour by max genus distance)
for (ni, nj), pd_val, phen_d in zip(pair_labels, phylo_dist_pairs, pheno_dist_pairs):
    same_genus = _genus(ni) == _genus(nj)
    color = '#2ecc71' if same_genus else '#e74c3c'
    ax.scatter(pd_val, phen_d, color=color, s=80, alpha=0.75,
               edgecolors='white', linewidths=0.6, zorder=3)
    ax.annotate(
        f'{ni.split()[0][0]}. {ni.split()[-1]} –\n{nj.split()[0][0]}. {nj.split()[-1]}',
        (pd_val, phen_d), xytext=(5, 2), textcoords='offset points',
        fontsize=6.5, color='#555555',
    )

# Regression line
m_coef = np.polyfit(phylo_dist_pairs, pheno_dist_pairs, 1)
x_line = np.linspace(phylo_dist_pairs.min(), phylo_dist_pairs.max(), 100)
ax.plot(x_line, np.polyval(m_coef, x_line),
        color='black', lw=1.5, ls='--', zorder=4, label=f'Linear fit  (r = {r:.2f})')

legend_patches = [
    mpatches.Patch(color='#2ecc71', label='Same genus'),
    mpatches.Patch(color='#e74c3c', label='Different genus'),
]
ax.legend(handles=legend_patches + ax.get_legend_handles_labels()[0][-1:],
          fontsize=8.5, framealpha=0.9)

ax.set_xlabel('Phylogenetic distance (patristic)', fontsize=10)
ax.set_ylabel('|Δ mean bloom DOY|  (days)', fontsize=10)
ax.set_title('Phylogenetic signal in flowering phenology\n'
             'Do closely related species flower at similar times?',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Pearson r (phylo dist vs |Δ bloom DOY|): {r:.3f}')

## 9. Clustermap — phenology × phylogeny

Hierarchically-clustered heatmap of species, with rows ordered by phylogenetic
dendrogram and columns representing quantile bins of year.  Shows whether
the phylogenetic ordering also reveals structure in multi-year bloom timing.

In [ ]:
# Annual median bloom DOY per species — only species with enough data
MIN_YEARS = 5
sp_annual = {}
for k, name in labels.items():
    sid = k[1]
    sub = df_bloom[df_bloom[KEY_SPECIES_ID] == sid].copy()
    ann = sub.groupby('year')['doy'].median()
    if len(ann) >= MIN_YEARS:
        sp_annual[name] = ann

all_years = sorted(set(y for ann in sp_annual.values() for y in ann.index))
heat_df = pd.DataFrame(
    {name: ann.reindex(all_years) for name, ann in sp_annual.items()},
    index=all_years,
).T

# Reorder rows by phylogenetic dendrogram (using leaves order from above)
dend_order = [label_list[i] for i in sch.leaves_list(linkage) if label_list[i] in heat_df.index]
heat_df = heat_df.loc[dend_order]

# Decade-averaged for cleaner display
decade_cols = {}
for col in heat_df.columns:
    decade = (col // 10) * 10
    decade_cols.setdefault(decade, []).append(col)
heat_decade = pd.DataFrame(
    {str(d) + 's': heat_df[cols].mean(axis=1) for d, cols in sorted(decade_cols.items())}
)

fig, ax = plt.subplots(figsize=(11, 4.5))

im = ax.imshow(heat_decade.values, cmap='RdYlGn_r', aspect='auto',
               vmin=np.nanpercentile(heat_decade.values, 5),
               vmax=np.nanpercentile(heat_decade.values, 95))
cbar = plt.colorbar(im, ax=ax, shrink=0.85)
cbar.set_label('Median bloom DOY', fontsize=9)

ax.set_yticks(range(len(heat_decade)))
ax.set_yticklabels(heat_decade.index, fontsize=9)
ax.set_xticks(range(len(heat_decade.columns)))
ax.set_xticklabels(heat_decade.columns, fontsize=9)

# Colour y-axis labels by genus
for ylabel_obj in ax.get_yticklabels():
    genus = _genus(ylabel_obj.get_text())
    ylabel_obj.set_color(genus_colors.get(genus, 'black'))
    ylabel_obj.set_fontweight('semibold')

ax.set_title(
    'Decade-mean bloom DOY — species ordered by phylogenetic dendrogram',
    fontsize=11, fontweight='bold'
)
ax.set_xlabel('Decade', fontsize=10)

# Genus legend on side
patches = [mpatches.Patch(color=c, label=g) for g, c in genus_colors.items()
           if any(_genus(sp) == g for sp in heat_decade.index)]
ax.legend(handles=patches, title='Genus', fontsize=8, title_fontsize=8.5,
          loc='lower right', framealpha=0.9)

plt.tight_layout()
plt.show()

## 10. Summary

Key findings from the phylogenetic analysis:

In [ ]:
# Print a clean summary table: species × (mean bloom DOY, std, phylo cluster)
# Assign cluster labels from the dendrogram (cut at half of max distance)
cut_height = linkage[-1, 2] * 0.5
cluster_ids = sch.fcluster(linkage, cut_height, criterion='distance')
cluster_map = {name: f'Cluster {c}' for name, c in zip(label_list, cluster_ids)}

rows = []
for name in label_list:
    sid = next(k[1] for k, n in labels.items() if n == name)
    doys = df_bloom.loc[df_bloom[KEY_SPECIES_ID] == sid, 'doy'].values
    rows.append({
        'Species': name,
        'Genus': _genus(name),
        'N obs': len(doys),
        'Mean bloom DOY': round(doys.mean(), 1) if len(doys) else float('nan'),
        'Std (days)': round(doys.std(), 1) if len(doys) else float('nan'),
        'Phylo cluster': cluster_map[name],
    })

summary_df = pd.DataFrame(rows).set_index('Species').sort_values('Mean bloom DOY')
print(summary_df.to_string())
print(f'\nPearson r (phylo dist vs |Δ bloom DOY|): {r:.3f}')